In [17]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine


# database connections 

In [18]:
with open('../config_fill.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['CO_SA']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)

# Extract

In [19]:
dim_mensajeria_estado = pd.read_sql_table('mensajeria_estado', co_sa)
dim_mensajeria_estado.rename(columns={'nombre': 'nombre','descripcion':'orden_secuencia'}, inplace=True)

# Transformations

In [20]:
dim_mensajeria_estado.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id               6 non-null      int64
 1   nombre           6 non-null      str  
 2   orden_secuencia  6 non-null      str  
dtypes: int64(1), str(2)
memory usage: 276.0 bytes


In [21]:
dim_mensajeria_estado.describe(include='all')

,id,nombre,orden_secuencia
count,6.000000,6,6
unique,NaN,6,6
top,NaN,Recogido por mensajero,Recogido por mensajero
freq,NaN,1,1
mean,3.500000,NaN,NaN
std,1.870829,NaN,NaN
min,1.000000,NaN,NaN
25%,2.250000,NaN,NaN
50%,3.500000,NaN,NaN
75%,4.750000,NaN,NaN


In [22]:
dim_mensajeria_estado.replace({np.nan: 'no aplica', ' ': 'no aplica','':'no_aplica'}, inplace=True)
#dim_mensajeria_estado["saved"] = date.today()
dim_mensajeria_estado.rename(columns={'id': 'id_estado','nombre': 'nombre_estado'}, inplace=True)

# load

In [23]:
dim_mensajeria_estado.head()

,id_estado,nombre_estado,orden_secuencia
0,4,Recogido por mensajero,Recogido por mensajero
1,5,Entregado en destino,Entregado en destino
2,3,Con novedad,Tiene novedad
3,6,Terminado completo,Terminado completo
4,1,Iniciado,Creado por el usuario


In [24]:
dim_mensajeria_estado.to_sql('dim_estado', etl_conn, if_exists='replace',index_label='key_dim_estado')

6